# Day 4, Part 3: From Similarity to Topics

**MIDAS Summer Academy**

In Part 1 you used pre-trained models off the shelf. In Part 2 you used a large language model to extract structured data from text. This notebook works with the representation underneath both: the **embedding**.

In this notebook you will:

1. Compute embeddings for individual sentences and measure the similarity between them with a **dot product**.
2. Embed a corpus of ~2,000 research abstracts.
3. Use **BERTopic** to discover topics in the corpus without labels or supervision.

---
## Setup

> **Before you start: enable a GPU runtime.** This notebook embeds ~2,000 abstracts, which runs much faster on a GPU. In Colab, go to **Runtime > Change runtime type**, set **Hardware accelerator** to **T4 GPU**, and click **Save**. The notebook still works on a CPU runtime, but the embedding steps will take minutes instead of seconds.

Run this once. The install takes a minute or two.

In [ ]:
!pip install -q bertopic sentence-transformers

In [ ]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=3, suppress=True)  # readable arrays
print("Ready.")

---
## 1. An embedding is an array of numbers

An **embedding model** maps a piece of text to a fixed-length vector. Embedding models are trained so that texts with similar meaning map to vectors that point in similar directions.

We will use `all-MiniLM-L6-v2`, a small sentence embedding model. It maps any sentence to a **384-dimensional** vector.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")   # ~80 MB download

In [ ]:
sentences = [
    "The cat sat on the warm windowsill.",
    "A kitten napped in the sunny window.",
    "The stock market fell sharply on Tuesday.",
    "Investors sold shares as prices dropped.",
    "I added fresh basil to the tomato sauce.",
]

embeddings = model.encode(sentences)
print("Shape:", embeddings.shape)

The next cell prints part of one embedding. It is a vector of 384 floating-point numbers.

In [ ]:
print("Embedding for:", sentences[0])
print("First 12 of its 384 numbers:")
print(embeddings[0][:12])

# These vectors are already normalized to length 1 (unit vectors).
print("\nLength (norm) of this vector:", np.linalg.norm(embeddings[0]).round(3))

### Measuring similarity with a dot product

For two vectors, the **dot product** is

$$a \cdot b = \sum_i a_i b_i$$

The embeddings from this model are normalized to length 1 (**unit vectors**), so the dot product between any two of them ranges from -1 to 1; higher values indicate greater similarity.

The next cell computes the dot product for every pair of sentences.

In [ ]:
# Compute the dot product for every pair of sentences
n = len(sentences)
similarity = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        similarity[i, j] = np.dot(embeddings[i], embeddings[j])
        labels = [f"S{i}" for i in range(n)]


Let's look at what comes out. Take a minute to decide: which pair of sentences has the highest dot product? Does this surprise you? 

In [ ]:
similarity

### Exercise 1

Add a sixth sentence to the `sentences` list. You could try a *paraphrase* of an existing sentence (different words, same meaning). Re-embed and rebuild the similarity matrix. How similar is your new sentence to the others? 

In [ ]:
# YOUR CODE HERE
# Copy the sentences list from above, append one new sentence, then re-embed
# and rebuild the similarity matrix.


---
## 2. Scaling up: 2,000 abstracts

The next dataset contains **~2,000 recent arXiv abstracts** drawn from six fields (astrophysics, machine learning, neuroscience, materials science, econometrics, atmospheric science). We can represent each with an embedding and discover topics with a widely used topic modeling library, BERTopic.

In [ ]:
import os

LOCAL = "arxiv_abstracts/abstracts.csv"
URL = "https://gist.githubusercontent.com/elleobrien/16b86d0447fbe49a5fce5172e670150c/raw/abstracts.csv"

df = pd.read_csv(LOCAL if os.path.exists(LOCAL) else URL)
print(f"Loaded {len(df)} abstracts.")
df["field"].value_counts()

The `field` column is the ground truth: the arXiv category each abstract came from. BERTopic does not use this column. We keep it to check, at the end, whether the discovered topics align with the actual fields.

---
## 3. Discover topics with BERTopic

**BERTopic** is a topic modeling library that chains four steps:

1. **Embed** every abstract, as in Section 1.
2. **Reduce** the 384 dimensions to a smaller number, preserving local structure (an algorithm called UMAP).
3. **Cluster** the reduced vectors; each cluster becomes a topic.
4. **Label** each cluster with the words most distinctive to it.

BERTopic runs all four steps internally with reasonable defaults. Step 1 is the same operation from Section 1: BERTopic calls the embedding model on every abstract to produce a 384-dimensional vector for each. In Section 1 we called `model.encode` ourselves; here BERTopic calls it for us. The [algorithm section of the BERTopic documentation](https://maartengr.github.io/BERTopic/algorithm/algorithm.html) has a visual walkthrough of each step.

In [ ]:
abstracts = df["abstract"].tolist()
print(f"{len(abstracts)} abstracts ready for BERTopic.")

### First pass: the defaults

Run BERTopic the way you would on a brand-new dataset: with its default settings. We pass two things:

- `embedding_model`: the model from Section 1, so BERTopic uses the same embeddings we studied there.
- `vectorizer_model`: removes common English words (stopwords) before choosing topic keywords, so the labels are built from meaningful terms.

Everything else is left at the defaults. In particular, BERTopic's default clustering algorithm decides **how many** topics there are on its own.

`fit_transform` runs all four steps in order. Embedding is the slowest step: roughly 30-60 seconds on a Colab CPU runtime, a few seconds on a GPU runtime. The dimensionality reduction step is stochastic, so the exact topics and their numbering can vary slightly between runs.

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

default_topic_model = BERTopic(
    embedding_model=model,
    vectorizer_model=CountVectorizer(stop_words="english"),
)

# BERTopic embeds the abstracts with `model`, then reduces, clusters, and labels them.
default_topics, _ = default_topic_model.fit_transform(abstracts)
print("Found", len(set(default_topics)), "topics.")

`get_topic_info()` lists every topic the model found, its size, and its top keywords:

In [ ]:
default_topic_model.get_topic_info()[["Topic", "Count", "Name"]]

The corpus was built from six fields, but the defaults typically find four topics here. Something got merged. Cross-tabulate the discovered topics against the ground-truth `field` column: each row is a discovered topic, each column is a true field.

In [ ]:
# Rows are discovered topics, columns are true fields.
pd.crosstab(pd.Series(default_topics, name="topic"), df["field"])

Three of the fields (astrophysics, materials science, atmospheric science) each map cleanly onto one topic. Machine learning, neuroscience, and econometrics were merged into a single large topic. Look at that topic's name in the table above: its keywords (model, data, framework) are the statistical vocabulary all three fields share. Their abstracts sit close together in embedding space with no clear gap between them, so the default clustering treats them as one group.

The number of topics is a modeling decision, and the defaults will not always match what you know about your data. Since we know this corpus was built from six fields, we can swap the default clusterer for **KMeans**, which partitions the documents into exactly the number of clusters we ask for.

### Second pass: setting the number of topics

The `hdbscan_model` argument is the clusterer BERTopic uses in step 3. It is named after the default algorithm (HDBSCAN), but it accepts any scikit-learn clusterer. We pass `KMeans(n_clusters=N_TOPICS)` to ask for exactly six topics.

In [ ]:
from sklearn.cluster import KMeans

N_TOPICS = 6

topic_model = BERTopic(
    embedding_model=model,
    hdbscan_model=KMeans(n_clusters=N_TOPICS),  # accepts any scikit-learn clusterer
    vectorizer_model=CountVectorizer(stop_words="english"),
)

topics, _ = topic_model.fit_transform(abstracts)
print("Found", len(set(topics)), "topics.")

### Inspect the topics

Six topics now, and the keywords make the split visible: the merged model/data topic from the first pass has separated into distinct topics.

In [ ]:
topic_model.get_topic_info()[["Topic", "Count", "Name"]]

With KMeans, every abstract is assigned to exactly one of the six topics. The next cell prints the defining words for a single topic:

In [ ]:
print("Top words for topic 0:")
for word, score in topic_model.get_topic(0):
    print(f"  {word:20s} {score:.3f}")

### Did the topics recover the fields?

Cross-tabulate again and compare with the cross-tabulation from the first pass. Are machine learning, neuroscience, and econometrics separated now? Look at where abstracts still land in a different topic than their field: which pairs of fields still get confused, and what about their subject matter might explain it?

In [ ]:
df["topic"] = topics
# Rows are discovered topics, columns are true fields.
crosstab = pd.crosstab(df["topic"], df["field"])
crosstab

---
## 4. Explore the topic space

BERTopic includes interactive visualizations. The **intertopic distance map** below projects the topics into two dimensions: each circle is a topic, circle size is the number of abstracts in it, and nearby circles contain semantically related abstracts.

In [ ]:
topic_model.visualize_topics()

### Search by meaning

`find_topics` embeds a query string and returns the topics whose contents are closest to it in embedding space. This is the same similarity measure from Section 1, applied as a search over topics.

In [ ]:
query = "exploding stars and supernovae"
found, scores = topic_model.find_topics(query, top_n=3)

print(f'Topics most similar to: "{query}"\n')
for t, s in zip(found, scores):
    words = ", ".join(w for w, _ in topic_model.get_topic(t)[:6])
    print(f"  Topic {t} (score {s:.2f}): {words}")

### Exercise 2

1. Call `find_topics` with a phrase from your own research area. Do the returned topics make sense?
2. Change `N_TOPICS` (try `4`, then `12`) and re-run Section 3. How do the topics change? Which fields merge when there are too few topics, and which split apart when there are too many?

In [ ]:
# YOUR CODE HERE
# query = "..."
# found, scores = topic_model.find_topics(query, top_n=3)


---
## Bonus: a science-specific embedding model

`all-MiniLM-L6-v2` was trained on general text from the web. **SPECTER** is an embedding model built for scientific papers. It starts from **SciBERT**, a BERT variant pre-trained on scientific text, and is fine-tuned on citation data: papers that cite each other are trained to have similar embeddings. It maps each document to a **768-dimensional** vector.

Re-embed the abstracts with SPECTER and rerun BERTopic with the same settings. The model is larger than MiniLM (~440 MB download), and embedding takes several minutes on a CPU runtime.

In [ ]:
specter = SentenceTransformer("sentence-transformers/allenai-specter")   # ~440 MB download

# Several minutes on a CPU runtime; much faster on a GPU runtime.
specter_embeddings = specter.encode(abstracts, show_progress_bar=True)
print("Embedded:", specter_embeddings.shape)

In [ ]:
specter_topic_model = BERTopic(
    embedding_model=specter,
    hdbscan_model=KMeans(n_clusters=N_TOPICS),
    vectorizer_model=CountVectorizer(stop_words="english"),
)

specter_topics, _ = specter_topic_model.fit_transform(abstracts, embeddings=specter_embeddings)
print("Found", len(set(specter_topics)), "topics.")
specter_topic_model.get_topic_info()[["Topic", "Count", "Name"]]

In [ ]:
# Rows are discovered topics, columns are true fields.
pd.crosstab(pd.Series(specter_topics, name="topic"), df["field"])

Compare this cross-tabulation with the one from Section 3. Look at the columns for econometrics, machine learning, and neuroscience: these three fields share a lot of statistical vocabulary, and they are where the two embedding models differ most. Does SPECTER separate them more cleanly than MiniLM, or less? Domain-specific pre-training changes which similarities the model emphasizes, so the answer depends on the corpus.